<a href="https://colab.research.google.com/github/iffathsaleem/RFM-Market-Analysis-Stock-Segmentation./blob/main/Market_Analysis_and_Stock_Segmentation_Using_RFM_CW.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install Yfinance

In [2]:
import pandas as pd
import numpy as np
import yfinance as yf
import matplotlib.pyplot as plt
import seaborn as sns

In [3]:
start = '2021-01-01'
end = '2025-01-01'
benchmark_ticker = '^GSPC'

from pathlib import Path

output_folder = Path('CRM_CM1_Results')
data_folder = output_folder / 'raw_data'
data_folder.mkdir(parents=True, exist_ok=True)

In [5]:
url = 'https://en.wikipedia.org/wiki/List_of_S%26P_500_companies'
ticker_file = data_folder/ 'SP500_Companies.csv'

if ticker_file.exists():
    sp500_df = pd.read_csv(ticker_file)
    print('Using the saved constituent list. Check ticker_source.csv for its date.')
else:
    tables = pd.read_html(url, storage_options={'User-Agent': 'Mozilla/5.0'})
    sp500_df = tables[0]
    assert {'Symbol', 'GICS Sector'}.issubset(sp500_df.columns), 'Check the Wikipedia table'
    sp500_df['Ticker'] = sp500_df['Symbol'].str.replace('.', '-', regex=False)
    sp500_df.to_csv(ticker_file, index=False)

    ticker_source = pd.DataFrame([{
        'Source': url,
        'RetrievedUTC': pd.Timestamp.now(tz='UTC').isoformat(),
        'Start': start,
        'EndExclusive': end
    }])
    ticker_source.to_csv(data_folder / 'ticker_source.csv', index=False)

tickers = sp500_df['Ticker'].tolist()
print('Total tickers:', len(tickers))
sp500_df[['Ticker', 'Security', 'GICS Sector']].head(10)

Total tickers: 503


,Ticker,Security,GICS Sector
0,MMM,3M,Industrials
1,AOS,A. O. Smith,Industrials
2,ABT,Abbott Laboratories,Health Care
3,ABBV,AbbVie,Health Care
4,ACN,Accenture,Information Technology
5,ADBE,Adobe Inc.,Information Technology
6,AMD,Advanced Micro Devices,Information Technology
7,AES,AES Corporation,Utilities
8,AFL,Aflac,Financials
9,A,Agilent Technologies,Health Care


In [6]:
import time

def download_data(ticker):
    filename = ticker.replace('^', 'index_') + '.csv'
    saved_file = data_folder / filename

    if saved_file.exists():
        data = pd.read_csv(saved_file, index_col=0, parse_dates=True)
    else:
        data = pd.DataFrame()
        for attempt in range(3):
            try:
                data = yf.Ticker(ticker).history(
                    start=start, end=end, auto_adjust=False, actions=True
                )
                if not data.empty:
                    break
            except Exception as error:
                print(ticker, error)
            time.sleep(2)

        if data.empty:
            raise ValueError('Download failed after three attempts')

        data.index = pd.to_datetime(data.index).tz_localize(None).normalize()
        data.to_csv(saved_file)

        download_record = pd.DataFrame([{
            'Ticker': ticker,
            'RetrievedUTC': pd.Timestamp.now(tz='UTC').isoformat(),
            'Start': start,
            'EndExclusive': end,
            'AutoAdjust': False
        }])
        download_record.to_csv(
            data_folder / (ticker.replace('^', 'index_') + '_source.csv'),
            index=False
        )

    data.index = pd.to_datetime(data.index).tz_localize(None).normalize()
    return data.sort_index()